In [1]:
import numpy as np
import open3d as o3d
import sklearn.neighbors as skln
from tqdm import tqdm
from scipy.io import loadmat
import multiprocessing as mp
import argparse

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
data_mesh = o3d.io.read_triangle_mesh('outputs/dtu_scan24/recon_aligned.ply')

In [6]:
vertices = np.asarray(data_mesh.vertices)

In [7]:
vertices

array([[  51.85290146,  -99.37555695,  588.71551514],
       [-113.29831696,  -37.04175186,  529.62823486],
       [  22.42683601, -151.32038879,  595.85791016],
       ...,
       [  23.5880928 , -142.2300415 ,  595.4901123 ],
       [  23.92084122, -141.58073425,  595.20861816],
       [  24.23740387, -140.97599792,  595.20861816]], shape=(716627, 3))

In [9]:
triangles = np.asarray(data_mesh.triangles)

In [10]:
triangles

array([[    35,     40,    234],
       [   280,    271,     51],
       [575814,      9,      0],
       ...,
       [712112,     32,     26],
       [347224,     32, 712112],
       [347224, 712112, 347016]], shape=(1420460, 3), dtype=int32)

In [18]:
tri_vert = vertices[triangles]

In [22]:
v1 = tri_vert[:,1] - tri_vert[:,0]

In [23]:
v2 = tri_vert[:,2] - tri_vert[:,0]

In [26]:
v1

array([[ 0.14511108,  0.        ,  0.27185059],
       [-0.19596291, -0.27563477,  0.        ],
       [-0.05587769, -0.07107544,  0.        ],
       ...,
       [-0.14524841,  0.        ,  0.64929199],
       [-0.16947556, -0.34642029,  0.        ],
       [-0.02422714, -0.34642029, -0.64929199]], shape=(1420460, 3))

In [25]:
v2

array([[ 0.14511108, -0.24098587,  0.64935303],
       [ 0.        , -0.27563477, -0.3314209 ],
       [-0.30485535, -0.07107544, -0.64929199],
       ...,
       [-0.44700623, -0.64932251,  0.64929199],
       [-0.02422714, -0.34642029, -0.64929199],
       [ 0.        , -0.31173706, -0.64929199]], shape=(1420460, 3))

In [27]:
l1 = np.linalg.norm(v1, axis=-1, keepdims=True)

In [31]:
l2 = np.linalg.norm(v2, axis=-1, keepdims=True)

In [32]:
l1

array([[0.30815575],
       [0.33819519],
       [0.09041036],
       ...,
       [0.66533991],
       [0.38565396],
       [0.7363247 ]], shape=(1420460, 1))

In [29]:
area2 = np.linalg.norm(np.cross(v1, v2), axis=-1, keepdims=True)

In [33]:
non_zero_area = (area2 > 0)[:,0]

In [34]:
non_zero_area

array([ True,  True,  True, ...,  True,  True,  True], shape=(1420460,))

In [37]:
area2

array([[0.09227973],
       [0.12442094],
       [0.06131203],
       ...,
       [0.47437283],
       [0.25540747],
       [0.02848892]], shape=(1420460, 1))

In [39]:
l1, l2, area2, v1, v2, tri_vert = [
            arr[non_zero_area] for arr in [l1, l2, area2, v1, v2, tri_vert]
        ]

In [45]:
thresh = 0.2

In [46]:
thr = thresh * np.sqrt(l1 * l2 / area2)

In [48]:
n1 = np.floor(l1 / thr)

In [51]:
n2 = np.floor(l2 / thr)

In [52]:
n2

array([[2.],
       [1.],
       [3.],
       ...,
       [4.],
       [3.],
       [0.]], shape=(1420446, 1))

In [72]:
n1

array([[1.],
       [1.],
       [0.],
       ...,
       [2.],
       [1.],
       [0.]], shape=(1420446, 1))

In [76]:
n1[1,0]

np.float64(1.0)

In [77]:
n2[1,0]

np.float64(1.0)

In [85]:
n1[10000,0]

np.float64(3.0)

In [74]:
c = np.mgrid[:n1[1,0]+1, :n2[1,0]+1]

In [78]:
c += 0.5 

In [79]:
c

array([[[0.5, 0.5],
        [1.5, 1.5]],

       [[0.5, 1.5],
        [0.5, 1.5]]])

In [82]:
c[0] /= max(n1[1,0], 1e-7)

In [84]:
c[1] /= max(n2[1,0], 1e-7)

In [86]:
def sample_single_tri(input_):
    n1, n2, v1, v2, tri_vert = input_
    c = np.mgrid[:n1+1, :n2+1]
    c += 0.5
    c[0] /= max(n1, 1e-7)
    c[1] /= max(n2, 1e-7)
    c = np.transpose(c, (1,2,0))
    k = c[c.sum(axis=-1) < 1]  # m2
    q = v1 * k[:,:1] + v2 * k[:,1:] + tri_vert
    return q

In [87]:
with mp.Pool() as mp_pool:
            new_pts = mp_pool.map(sample_single_tri, ((n1[i,0], n2[i,0], v1[i:i+1], v2[i:i+1], tri_vert[i:i+1,0]) for i in range(len(n1))), chunksize=1024)

In [90]:
new_pts = np.concatenate(new_pts, axis=0)

In [92]:
data_pcd = np.concatenate([vertices, new_pts], axis=0)

In [101]:
data_pcd

array([[  51.85290146,  -99.37555695,  588.71551514],
       [-113.29831696,  -37.04175186,  529.62823486],
       [  22.42683601, -151.32038879,  595.85791016],
       ...,
       [  41.42886543, -202.37242508,  622.39845276],
       [  41.57974434, -202.04776382,  622.39845276],
       [  41.68000793, -201.85112508,  622.37139893]], shape=(3395793, 3))

In [94]:
nn_engine = skln.NearestNeighbors(n_neighbors=1, radius=thresh, algorithm='kd_tree', n_jobs=-1)

In [96]:
nn_engine.fit(data_pcd)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",0.2
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'kd_tree'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


In [97]:
rnn_idxs = nn_engine.radius_neighbors(data_pcd, radius=thresh, return_distance=False)

In [98]:
rnn_idxs

array([array([2439788,       0,  716648]),
       array([      1,  716639, 2285027]),
       array([      2,  655848, 3373255]), ..., array([3395790]),
       array([3395791,      32]), array([     32, 3395792])],
      shape=(3395793,), dtype=object)

In [99]:
mask = np.ones(data_pcd.shape[0], dtype=np.bool_)

In [100]:
for curr, idxs in enumerate(rnn_idxs):
        if mask[curr]:
            mask[idxs] = 0
            mask[curr] = 1

array([ True,  True,  True, ...,  True,  True,  True], shape=(3395793,))

In [102]:
data_down = data_pcd[mask]

In [103]:
data_down

array([[  51.85290146,  -99.37555695,  588.71551514],
       [-113.29831696,  -37.04175186,  529.62823486],
       [  22.42683601, -151.32038879,  595.85791016],
       ...,
       [  41.42886543, -202.37242508,  622.39845276],
       [  41.57974434, -202.04776382,  622.39845276],
       [  41.68000793, -201.85112508,  622.37139893]], shape=(3395793, 3))

In [105]:
obs_mask_file = loadmat(f'dtu_eval/Offical_DTU_Dataset/ObsMask/ObsMask24_10.mat')

In [107]:
ObsMask, BB, Res = [obs_mask_file[attr] for attr in ['ObsMask', 'BB', 'Res']]

In [114]:
BB

array([[-198., -225.,  473.],
       [ 186.,  265.,  870.]], dtype=float32)

In [112]:
BB = BB.astype(np.float32)

In [113]:
patch = 60

In [115]:
inbound = ((data_down >= BB[:1]-patch) & (data_down < BB[1:]+patch*2)).sum(axis=-1) ==3

In [116]:
inbound

array([ True,  True,  True, ...,  True,  True,  True], shape=(3395793,))

In [117]:
data_in = data_down[inbound]

In [118]:
data_grid = np.around((data_in - BB[:1]) / Res).astype(np.int32)

In [120]:
grid_inbound = ((data_grid >= 0) & (data_grid < np.expand_dims(ObsMask.shape, 0))).sum(axis=-1) ==3

In [122]:
data_grid_in = data_grid[grid_inbound]

In [123]:
in_obs = ObsMask[data_grid_in[:,0], data_grid_in[:,1], data_grid_in[:,2]].astype(np.bool_)

In [124]:
data_in_obs = data_in[grid_inbound][in_obs]

In [125]:
data_in_obs

array([[  51.85290146,  -99.37555695,  588.71551514],
       [-113.29831696,  -37.04175186,  529.62823486],
       [  22.42683601, -151.32038879,  595.85791016],
       ...,
       [ -44.7849884 ,   24.96738815,  576.91967773],
       [ -44.80250041,   24.7509524 ,  576.70324707],
       [ -44.4988327 ,   25.18662024,  576.50146484]], shape=(2644322, 3))

In [126]:
import pickle

In [128]:
with open('eval_tnt/GT_TNT_dataset/poses_data.pkl', 'rb') as file:
# Load the object from the file
    poses = pickle.load(file)

In [24]:
path = 'eval_tnt/GT_TNT_dataset/Barn/sparse/cameras.txt'

In [25]:
with open(path, 'r', encoding='utf-8') as file:
    content = file.read()
    print(content)

1 RADIAL 1920 1080 1152.0 960.0 540.0 0 0



In [10]:
def read_next_bytes(fid, num_bytes, format_char_sequence, endian_character="<"):
    """Read and unpack the next bytes from a binary file.
    :param fid:
    :param num_bytes: Sum of combination of {2, 4, 8}, e.g. 2, 6, 16, 30, etc.
    :param format_char_sequence: List of {c, e, f, d, h, H, i, I, l, L, q, Q}.
    :param endian_character: Any of {@, =, <, >, !}
    :return: Tuple of read and unpacked values.
    """
    data = fid.read(num_bytes)
    return struct.unpack(endian_character + format_char_sequence, data)

In [8]:
def read_intrinsics_binary(path_to_model_file):
    """
    see: src/base/reconstruction.cc
        void Reconstruction::WriteCamerasBinary(const std::string& path)
        void Reconstruction::ReadCamerasBinary(const std::string& path)
    """
    cameras = {}
    with open(path_to_model_file, "rb") as fid:
        num_cameras = read_next_bytes(fid, 8, "Q")[0]
        for _ in range(num_cameras):
            camera_properties = read_next_bytes(
                fid, num_bytes=24, format_char_sequence="iiQQ")
            camera_id = camera_properties[0]
            model_id = camera_properties[1]
            model_name = CAMERA_MODEL_IDS[camera_properties[1]].model_name
            width = camera_properties[2]
            height = camera_properties[3]
            num_params = CAMERA_MODEL_IDS[model_id].num_params
            params = read_next_bytes(fid, num_bytes=8*num_params,
                                     format_char_sequence="d"*num_params)
            cameras[camera_id] = Camera(id=camera_id,
                                        model=model_name,
                                        width=width,
                                        height=height,
                                        params=np.array(params))
        assert len(cameras) == num_cameras
    return cameras

In [17]:
import numpy as np
import collections

In [12]:
import struct

In [18]:
CameraModel = collections.namedtuple(
    "CameraModel", ["model_id", "model_name", "num_params"])
Camera = collections.namedtuple(
    "Camera", ["id", "model", "width", "height", "params"])
BaseImage = collections.namedtuple(
    "Image", ["id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids"])
Point3D = collections.namedtuple(
    "Point3D", ["id", "xyz", "rgb", "error", "image_ids", "point2D_idxs"])

In [19]:
CAMERA_MODELS = {
    CameraModel(model_id=0, model_name="SIMPLE_PINHOLE", num_params=3),
    CameraModel(model_id=1, model_name="PINHOLE", num_params=4),
    CameraModel(model_id=2, model_name="SIMPLE_RADIAL", num_params=4),
    CameraModel(model_id=3, model_name="RADIAL", num_params=5),
    CameraModel(model_id=4, model_name="OPENCV", num_params=8),
    CameraModel(model_id=5, model_name="OPENCV_FISHEYE", num_params=8),
    CameraModel(model_id=6, model_name="FULL_OPENCV", num_params=12),
    CameraModel(model_id=7, model_name="FOV", num_params=5),
    CameraModel(model_id=8, model_name="SIMPLE_RADIAL_FISHEYE", num_params=4),
    CameraModel(model_id=9, model_name="RADIAL_FISHEYE", num_params=5),
    CameraModel(model_id=10, model_name="THIN_PRISM_FISHEYE", num_params=12)
}

In [20]:
CAMERA_MODEL_IDS = dict([(camera_model.model_id, camera_model)
                         for camera_model in CAMERA_MODELS])
CAMERA_MODEL_NAMES = dict([(camera_model.model_name, camera_model)
                           for camera_model in CAMERA_MODELS])

In [21]:
text = read_intrinsics_binary(path)

In [22]:
text

{1: Camera(id=1, model='PINHOLE', width=1918, height=1079, params=array([1152. , 1152. ,  959. ,  539.5]))}